# Adult Census Analysis and PCA

**Author:** Arian Abdollahi · **Course:** Data Mining

Portable copy of the submitted coursework. See `../README.md` and `../reports/results.md`. Historical outputs are retained in `../submission/`; outputs here are cleared before execution.


In [ ]:
from pathlib import Path

# Find this project's data directory from a repository, project, or notebook cwd.
PROJECT_SLUG = '01-adult-income-pca'
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next(
    (candidate for parent in SEARCH_ROOTS
     for candidate in (parent, parent / "projects" / PROJECT_SLUG)
     if candidate.name == PROJECT_SLUG and (candidate / "data").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Start Jupyter from this repository or project directory.")
DATA_DIR = PROJECT_DIR / "data"


# Import necessary libraries:

In [ ]:
# آرین عبداللهی 40023055
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder


# Load and Preprocess dataset

In [ ]:
# Load the dataset using pandas.read_csv

df1 = pd.read_csv(DATA_DIR / "adult.csv")

In [ ]:
# Print first rows the dataset
print(df1.iloc[:1]) 

In [ ]:
# Drop rows with any missing values
for i in df1.columns:
    df1 = df1.drop(df1[df1[i] == '?'].index) # drop any row from our dataset that contain "?" in columns  
m=df1.head(15)
df= df1.reset_index(drop=True)
df


# Visualize dataset

In [ ]:
# Visualize the dataset using scatter plots, histograms, box plots, and pair plots to understand the data distribution, identify outliers, and detect patterns.


# Scatter plot for Age vs Hours-per-week

# your code here:



plt.show()
x=df["hours-per-week"]
y=df["age"]
plt.scatter(x,y,c='b')
plt.xlabel('hours-per-week')
plt.ylabel('age')


# Histogram for distribution of age distribution

# your code here:

plt.show()
plt.hist(y,bins=10,edgecolor='black')
plt.xlabel('age')
plt.ylabel('Frequency')

# # Box plot for age to see outliers

# # your code here:

q1 = np.quantile(y, .25)  # calculate quantile 1 
q2 = np.quantile(y, .5)   # calculate quantile 2
q3 = np.quantile(y, .75)  # calculate quantile 3
iqr = q3 - q1             # calculate iqr
# identify outliers
lv = q1 - 1.5 * iqr       
hv = q3 + 1.5 * iqr

plt.show()
plt.boxplot(y)


# # Pair plot for Age, education-num, and hours-per-week

# # your code here:
g=sns.pairplot(
        df,
    x_vars=['age','educational-num','hours-per-week'],
    y_vars=['age','educational-num','hours-per-week'],
    corner=True,
    height=2.5,
    
    
)





# Encode and Normalize dataset

In [ ]:
# Use one-hot endcoding to convert categorical features into numerical format

categorical_columns = df.select_dtypes(include=['object']).columns.tolist()

# This line selects all columns from the DataFrame 'df' that have the data type 'object' (typically strings).
# Then, it extracts the column names from the result (a pandas Index object) and converts them to a regular Python list using .tolist()

# 2. Create a One-Hot Encoder object
encoder = OneHotEncoder(sparse_output=False)

# 3. Apply One-Hot Encoding
one_hot_encoded = encoder.fit_transform(df[categorical_columns])

# This line fits the encoder to the data in the selected categorical columns from 'df'. It then transforms the data using one-hot encoding and stores the result in 'one_hot_encoded'.

# 4. Create a DataFrame from the encoded data
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(categorical_columns))

# This line creates a new DataFrame 'one_hot_df' from the one-hot encoded data in 'one_hot_encoded'.
# The columns are named using the encoder's 'get_feature_names_out' method, which provides the names for the encoded features based on the original categorical column names and categories.

# 5. Combine original and encoded data
df_encoded = pd.concat([df, one_hot_df], axis=1)

# This line combines the original DataFrame 'df' with the one-hot encoded DataFrame 'one_hot_df' by concatenating them along axis=1 (columns).

# 6. Drop original categorical columns
df_encoded = df_encoded.drop(categorical_columns, axis=1)

# This line drops the original categorical columns (those selected in step 1) from the combined DataFrame 'df_encoded'. This is done because the information is now encoded in the new one-hot encoded columns.

df_encoded



In [ ]:
# Use StandardScaler from sklearn.preprocessing to normalize the features.

scaler = StandardScaler()

# your code here:
pca_imp=scaler.fit_transform(df_encoded) 
pca_imp

# Implement PCA from Scratch

In [ ]:
# Start by ensuring all data is numeric and normalized.
# Subtract the mean to center the data, calculate the covariance matrix, find eigenvalues and eigenvectors, sort them, and project the data onto the principal components.
# This section will test your understanding of PCA's mathematical foundation.


# your code here:

mean_vec = np.mean(pca_imp, axis=0)
cov_mat = (pca_imp - mean_vec).T.dot((pca_imp - mean_vec)) / (pca_imp.shape[0]-1) # calculte covariance matrix with formula 
print('Covariance matrix \n%s' %cov_mat)

cov_matrix = np.cov(pca_imp.T)        # calculte covariance matrix with numpy functions
eigen_values, eigen_vectors = np.linalg.eigh(cov_matrix)

# sort eigen values and eigen vectors
sorted_index = np.argsort(eigen_values)[::-1] 
sorted_eigenvalue = eigen_values[sorted_index]
sorted_eigenvectors = eigen_vectors[:,sorted_index]

# choose 10 principal components
k=10
principal_components = sorted_eigenvectors[:,:k]
pca_imp_new = np.dot(pca_imp, principal_components)
pca_imp_new


# Visualize the PCA-reduced data

In [ ]:

# Scatter Plot for 2D PCA Data (choose 2 pricipal components)
import matplotlib.pyplot as plt
# your code here:
k=2
principal_components = sorted_eigenvectors[:,:k]
pca_imp_new = np.dot(pca_imp, principal_components)


plt.scatter(pca_imp_new[:,0],pca_imp_new[:,1],c='b')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Principal Components')
plt.show()



# Heatmap for Principal Component Correlation
# This requires calculating the correlation matrix between the original data and the PCA-transformed data.
correlation_matrix1 = np.corrcoef(pca_imp_new.T)

sns.heatmap(correlation_matrix1, annot=True, cmap='coolwarm')
plt.title('Correlation between Original Features and Principal Components')
plt.xlabel('Principal Components')
plt.ylabel('Original Features')
plt.show()

correlation_matrix2 = np.corrcoef(df_encoded.T,pca_imp_new.T)

sns.heatmap(correlation_matrix2, annot=True, cmap='coolwarm')
plt.title('Correlation between Original Features and Principal Components')
plt.xlabel('Principal Components')
plt.ylabel('Original Features')
plt.show()




